# CT Conversion and Scan Inventory

An inventory and HDF5 inspection recipe for the default Basalt processing layout. `CT/processed` is an ordinary HDF5 output tree; the notebook does not assume it is a BagIt package.

Set `data_root` in your local configuration to the directory containing the Basalt data folders.

- `CT/raw/{datetime-of-scan}/` — BagIt folders
- `CT/processed/{datetime-of-scan}/{datetime-of-scan}_{condition}.hdf5` — processed CT volumes; the filename timestamp exactly matches its parent folder
- `CT/pre-test/raw/` and `CT/pre-test/processed/` — pre-test CT acquisition and processed volume
- `fluid-state/`, `resistance/`, and `pump/` — BagIt folders


In [ ]:
from pathlib import Path
import sys

working_directory = Path.cwd().resolve()
project_root = next(
    (
        candidate
        for candidate in (working_directory, *working_directory.parents)
        if (candidate / "pyproject.toml").is_file()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Run this notebook from within the basalt-processing project.")

source_root = project_root / "src"
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

from basalt_processing.paths import load_config, resolve_path

CONFIG_PATH = project_root / "config" / "basalt.example.toml"
config = load_config(CONFIG_PATH)
paths_config = config.get("paths", {})
config_dir = config.get("_config_dir")
data_root = resolve_path(paths_config["data_root"], config_dir)
data_paths = {
    name: resolve_path(paths_config.get(key, fallback), data_root)
    for name, key, fallback in (
        ("ct_raw_bagit", "ct_raw", "CT/raw"),
        ("ct_processed", "ct_processed", "CT/processed"),
        ("ct_pretest_raw", "ct_pretest_raw", "CT/pre-test/raw"),
        ("ct_pretest_processed", "ct_pretest_processed", "CT/pre-test/processed"),
        ("fluid_state_bagit", "fluid_state", "fluid-state"),
        ("resistance_bagit", "resistance", "resistance"),
        ("pump_bagit", "pump", "pump"),
    )
}
ct_processed_root = data_paths["ct_processed"]
data_paths


In [ ]:
import pandas as pd

processed_root = ct_processed_root
hdf5_files = sorted(processed_root.rglob("*.hdf5")) if processed_root.exists() else []
ct_inventory = pd.DataFrame(
    {
        "relative_path": [path.relative_to(processed_root).as_posix() for path in hdf5_files],
        "file_name": [path.name for path in hdf5_files],
        "timestamp": [path.parent.name for path in hdf5_files],
        "is_global": [path.name.endswith("_global.hdf5") or "._global." in path.name for path in hdf5_files],
    }
)
ct_inventory.head()


In [ ]:
from basalt_processing.hdf5_inspection import (
    describe_hdf5,
    find_pickled_amira_headers,
    raw_hdf5_attribute,
    render_amira_header,
    unpickle_amira_header,
)


In [ ]:
if hdf5_files:
    selected_hdf5 = hdf5_files[0]
    describe_hdf5(selected_hdf5)
else:
    print("No HDF5 files found under CT/processed.")


## Amira pickled-header attributes

Some Amira conversion workflows store a pickled header in an HDF5 attribute. The structure summary above identifies that attribute without printing its serialized value. The final cell unpickles and displays it for a trusted, local HDF5 file.


In [ ]:
if hdf5_files:
    selected_hdf5 = hdf5_files[0]
    header_attributes = find_pickled_amira_headers(selected_hdf5)
    if header_attributes:
        for object_path, attribute_name in header_attributes:
            print(f"Amira header: {object_path} / {attribute_name}")
            header = unpickle_amira_header(selected_hdf5, object_path, attribute_name)
            print(render_amira_header(header))
    else:
        print("No pickled Amira-header attributes found.")
else:
    print("No HDF5 files found under CT/processed.")


## HDF5 to Amira export

The notebook export below is opt-in because CT volumes can be large. It writes the selected HDF5 volume to `output/amira-exports/` rather than beside the processed source. Set `RUN_AMIRA_EXPORT = True` only when you want to create the `.am` file.

The exporter uses the selected dataset's voxel-size attributes. If a derived dataset such as `data_masked` has no voxel-size metadata, it silently uses the `data` dataset's voxel size so that the exported Amira bounding box retains the CT scale.

### Command line

From the project root, export one HDF5 file with:

```powershell
uv run basalt-convert-hdf5-to-amira path\to\volume.hdf5 output\volume.am
```

Export a processed directory recursively with:

```powershell
uv run basalt-convert-hdf5-to-amira CT\processed --output-dir output\amira-exports --recursive
```

Use `--dataset data` to choose a specific dataset, or `--dtype uint16` to cast before export.


In [ ]:
from basalt_processing.hdf5_amira import hdf5_to_amira

exported_am = None
RUN_AMIRA_EXPORT = True
if not hdf5_files:
    print("No HDF5 files found under CT/processed.")
elif not RUN_AMIRA_EXPORT:
    print("Set RUN_AMIRA_EXPORT = True to export the selected HDF5 file.")
else:
    selected_hdf5 = hdf5_files[0]
    output_am = data_root / "output" / "amira-exports" / f"{selected_hdf5.stem}.am"
    exported_am = hdf5_to_amira(selected_hdf5, output_am)
    print(f"Exported {selected_hdf5.name} to {exported_am}")


In [ ]:
from basalt_processing.amira_hdf5 import am_to_hdf5

RUN_AMIRA_IMPORT = False
if not RUN_AMIRA_IMPORT:
    print("Set RUN_AMIRA_IMPORT = True after exporting an Amira file.")
elif exported_am is None or not exported_am.is_file():
    print("Run the HDF5-to-Amira export cell first.")
else:
    roundtrip_h5 = (
        data_root / "output" / "amira-roundtrip" / f"{selected_hdf5.stem}.roundtrip.hdf5"
    )
    if roundtrip_h5.resolve() == selected_hdf5.resolve():
        raise ValueError("Round-trip output must be separate from the source HDF5.")
    am_to_hdf5(exported_am, roundtrip_h5)
    print(f"Recreated HDF5 volume at {roundtrip_h5}")